# Deep learning for time series (DeepAR, N-BEATS, TFT) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def make_series(n=60, trend=0.08, amp=2.0, noise=0.25):
    t=np.arange(n)
    return t, 10 + trend*t + amp*np.sin(2*np.pi*t/12) + noise*np.random.randn(n)
def acf(x, max_lag):
    x=np.asarray(x)-np.mean(x); den=np.dot(x,x)
    return np.array([1.0]+[np.dot(x[:-k],x[k:])/den for k in range(1,max_lag+1)])
def moving_average(x,w):
    return np.convolve(x, np.ones(w)/w, mode='valid')
print('setup ok')

## Neural forecasters learn functions from windows, covariates, and shared examples instead of hand-specifying every dependency
Deep time-series models turn history into supervised windows. These examples show a windowed prediction, shared learning across series, N-BEATS backcast/forecast, quantile output, and attention-style variable weighting.

In [ ]:
# 1) window-to-forecast map: a learned weighted average of recent values
x=np.array([10,12,14]); w=np.array([0.2,0.3,0.5]); pred=float(np.dot(w,x))
plt.figure(figsize=(6,3)); plt.bar(['old','mid','recent'],x*w); plt.title(f'weighted window prediction = {pred:.1f}')
assert abs(pred-12.6)<1e-12

In [ ]:
# 2) shared model pools multiple related series
series=np.vstack([10+np.arange(6),20+np.arange(6),30+np.arange(6)]); diffs=np.diff(series,axis=1); shared_slope=diffs.mean(); pred=series[:,-1]+shared_slope
plt.figure(figsize=(6,3)); plt.plot(series.T); plt.scatter([6,6,6],pred,c='r'); plt.title('one shared slope forecasts all series')
assert shared_slope==1.0 and np.allclose(pred,[16,26,36])

In [ ]:
# 3) N-BEATS block explains the past and projects the future
y_back=np.array([10,12,11]); backcast=np.array([10,11,12]); forecast=np.array([13,14]); back_mse=np.mean((y_back-backcast)**2)
plt.figure(figsize=(6,3)); plt.plot(range(3),y_back,'o-',label='past'); plt.plot(range(3),backcast,'s--',label='backcast'); plt.plot([3,4],forecast,'o-',label='forecast'); plt.legend(); plt.title('block has backcast and forecast heads')
assert abs(back_mse-0.6666666666666666)<1e-12

In [ ]:
# 4) probabilistic neural forecasts output quantiles, not just a mean
mean=np.array([10,11,12,13]); spread=np.array([1,1.2,1.4,1.6]); q10=mean-1.2816*spread; q90=mean+1.2816*spread
plt.figure(figsize=(6,3)); plt.plot(mean,label='median'); plt.fill_between(range(4),q10,q90,alpha=.25,label='80% band'); plt.legend(); plt.title('DeepAR-style forecast distribution')
assert np.all(q90-q10 > 2.5)

In [ ]:
# 5) TFT-style gating weights useful covariates more heavily
cov=np.array([0.2,1.5,-0.3]); logits=np.array([0.0,2.0,-1.0]); weights=np.exp(logits)/np.exp(logits).sum(); contribution=weights*cov
plt.figure(figsize=(6,3)); plt.bar(['price','promo','weather'],contribution); plt.title('variable selection weights contributions')
assert weights[1] > 0.8 and contribution[1] > contribution[0]